In [ ]:
import os
import requests
from typing import List
from pydantic import BaseModel
from pydantic_ai import Agent

# Note: I have to set an env variable for the API key, for now its hardcoded.
os.environ["GOOGLE_API_KEY"] = ""

class PaperRecommendation(BaseModel):
    paper_title: str
    authors: List[str]
    publication_year: int
    citation_count: int
    citation_source: str
    paper_link: str
    relevance_explanation: str


agent = Agent(
    "google:gemini-3.8-flash",
    output_type=PaperRecommendation, 
    system_prompt=(
        "You are an expert AI research assistant. Your job is to find academic papers "
        "that strictly match the user's constraints (topic, publication year, citation count). "
        "Use the search_academic_papers tool to fetch real data. If the initial results do "
        "not meet the constraints, search again or evaluate different papers until you find "
        "a matching paper."
    ),
)

@agent.tool_plain
def search_academic_papers(query: str) -> list[dict]:
    """
    Searches the Semantic Scholar database for academic papers.
    Use this to find papers by topic and check their year and citation count.
    """
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    params = {
        "query": query,
        "fields": "title,authors,year,citationCount,url",
        "limit": 10
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json().get("data", [])
        
        formatted_results = []
        for paper in data:
            author_names = [author["name"] for author in paper.get("authors", [])]
            formatted_results.append({
                "title": paper.get("title"),
                "authors": author_names,
                "year": paper.get("year"),
                "citationCount": paper.get("citationCount"),
                "url": paper.get("url")
            })
        return formatted_results
    return [{"error": "Search failed. Try a different query."}]

user_prompt = (
    "Find a research paper about LLM agents for software engineering that was "
    "published after 2022 and has at least 100 (>100) citations. Explain why the paper "
    "is relevant and provide the source of the citation count."
)

result = await agent.run(user_prompt)


print(f"Title: {result.output.paper_title}")
print(f"Authors: {', '.join(result.output.authors)}")
print(f"Year: {result.output.publication_year}")
print(f"Citations: {result.output.citation_count} (Source: {result.output.citation_source})")
print(f"Link: {result.output.paper_link}")
print(f"Explanation: {result.output.relevance_explanation}")

UserError: Set the `GOOGLE_API_KEY` environment variable or pass it via `GoogleProvider(api_key=...)` to use the Gemini API. To try Pydantic AI without an API key, use the built-in test model: `Agent('test')`. See https://pydantic.dev/docs/ai/guides/testing/